# 🛡️ GateGuard AI — Google Colab 추론 노트북

**동작 순서**
1. 패키지 설치
2. 구글 드라이브 마운트
3. 설정 (영상 경로 · 라인 좌표 등)
4. 추론 실행 → 결과 영상 저장

> GPU 런타임 권장: 런타임 → 런타임 유형 변경 → T4 GPU

## 1️⃣ 패키지 설치

In [ ]:
!pip install -q ultralytics==8.4.31 supervision==0.27.0.post2 opencv-python-headless==4.13.0.92 httpx==0.28.1
print('✅ 패키지 설치 완료')

## 2️⃣ 구글 드라이브 마운트

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print('✅ 드라이브 마운트 완료')

## 3️⃣ 설정

아래 경로와 파라미터를 본인 환경에 맞게 수정하세요.

In [ ]:
import os, glob

# ──────────────────────────────────────────────
# 드라이브 폴더 경로
# ──────────────────────────────────────────────
BASE_DIR    = '/content/drive/MyDrive/ㅈㅎㅊ'
CONVERT_DIR = f'{BASE_DIR}/변환 완료'
OUTPUT_DIR  = f'{BASE_DIR}/output'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# 사용할 카테고리 (emergencydoor 제외)
USE_CATEGORIES = ['unpaid', 'tailgating', 'nomal', 'jump', 'crawling']

# 해당 폴더들에서 영상 파일 수집
video_files = []
for cat in USE_CATEGORIES:
    cat_dir = f'{CONVERT_DIR}/{cat}'
    files = sorted(
        glob.glob(f'{cat_dir}/**/*.mp4', recursive=True) +
        glob.glob(f'{cat_dir}/**/*.avi', recursive=True)
    )
    video_files.extend(files)
    print(f'  {cat}: {len(files)}개')

print(f'\n📂 총 영상 파일: {len(video_files)}개')
for i, p in enumerate(video_files):
    print(f'  [{i:>3}] {os.path.relpath(p, CONVERT_DIR)}')

# ──────────────────────────────────────────────
# 분석할 영상 선택 (위 목록에서 인덱스 지정)
# 전체 처리하려면 아래 루프 셀에서 video_files 전체 순회
# ──────────────────────────────────────────────
VIDEO_INDEX = 0                          # ← 단일 영상 테스트 시 변경
VIDEO_PATH  = video_files[VIDEO_INDEX]
OUTPUT_PATH = f'{OUTPUT_DIR}/{os.path.splitext(os.path.basename(VIDEO_PATH))[0]}_result.mp4'
print(f'\n▶ 선택된 영상: {VIDEO_PATH}')
print(f'▶ 결과 저장:   {OUTPUT_PATH}')

# ──────────────────────────────────────────────
# 감지 라인 좌표 (영상 해상도에 맞게 조정 필요)
# ──────────────────────────────────────────────
LINE_START = (0, 360)
LINE_END   = (640, 360)

# ──────────────────────────────────────────────
# 추론 파라미터
# ──────────────────────────────────────────────
CONFIDENCE_THRESHOLD = 0.5
MAX_PREVIEW_FRAMES   = 200  # None = 전체 처리

REPORT_TO_BACKEND = False
BACKEND_URL       = 'http://localhost:8000'
SECRET_KEY        = 'gateguard-secret-key-dev'

print('\n✅ 설정 완료')

## 4️⃣ 모듈 정의

In [ ]:
import cv2
import numpy as np
import supervision as sv
from ultralytics import YOLO
from IPython.display import display, clear_output
from PIL import Image
import io


# ─── 얼굴 비식별화 ───────────────────────────────
class FaceAnonymizer:
    def __init__(self, model_path: str = 'yolov8n-face.pt'):
        try:
            self.model = YOLO(model_path)
            self._enabled = True
            print(f'✅ 얼굴 비식별화 모델 로드: {model_path}')
        except Exception as e:
            print(f'⚠️ 얼굴 비식별화 모델 로드 실패 ({e}) — 비활성화')
            self.model = None
            self._enabled = False

    def blur(self, frame: np.ndarray) -> np.ndarray:
        if not self._enabled:
            return frame
        results = self.model(frame, verbose=False)[0]
        for box in results.boxes.xyxy.cpu().numpy().astype(int):
            x1, y1, x2, y2 = box
            roi = frame[y1:y2, x1:x2]
            if roi.size == 0:
                continue
            frame[y1:y2, x1:x2] = cv2.GaussianBlur(roi, (51, 51), 0)
        return frame


# ─── 다중 객체 추적 ───────────────────────────────
class PersonTracker:
    def __init__(self):
        self.tracker        = sv.ByteTrack()
        self.annotator      = sv.BoxAnnotator()
        self.label_annotator = sv.LabelAnnotator()

    def update(self, detections: sv.Detections) -> sv.Detections:
        return self.tracker.update_with_detections(detections)

    def annotate(self, frame: np.ndarray, detections: sv.Detections) -> np.ndarray:
        labels    = [f'ID:{tid}' for tid in (detections.tracker_id or [])]
        annotated = self.annotator.annotate(frame.copy(), detections=detections)
        return self.label_annotator.annotate(annotated, detections=detections, labels=labels)


# ─── Colab 미리보기 유틸 ─────────────────────────
def show_frame(frame_bgr: np.ndarray):
    """BGR 프레임을 Colab 셀에 인라인 출력"""
    rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
    img = Image.fromarray(rgb)
    buf = io.BytesIO()
    img.save(buf, format='JPEG', quality=80)
    buf.seek(0)
    clear_output(wait=True)
    display(Image.open(buf))


print('✅ 모듈 정의 완료')

## 5️⃣ 추론 실행

In [ ]:
import os, datetime

# ─── 모델 로드 ────────────────────────────────────
person_model = YOLO('yolo11n.pt')   # 사람 감지
anonymizer   = FaceAnonymizer()     # 얼굴 비식별화
tracker      = PersonTracker()      # 추적

line_zone        = sv.LineZone(
    start=sv.Point(*LINE_START),
    end=sv.Point(*LINE_END),
)
line_annotator   = sv.LineZoneAnnotator()
triggered_ids    = set()
event_log        = []               # (frame_idx, track_id, confidence)

# ─── 영상 열기 ────────────────────────────────────
cap = cv2.VideoCapture(VIDEO_PATH)
if not cap.isOpened():
    raise FileNotFoundError(f'영상을 열 수 없습니다: {VIDEO_PATH}')

fps    = cap.get(cv2.CAP_PROP_FPS) or 30
width  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
total  = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
print(f'📹 영상 정보: {width}×{height} @ {fps:.1f}fps, 총 {total}프레임')

# ─── 출력 영상 준비 ───────────────────────────────
os.makedirs(os.path.dirname(OUTPUT_PATH), exist_ok=True)
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
writer = cv2.VideoWriter(OUTPUT_PATH, fourcc, fps, (width, height))

# ─── 프레임 루프 ─────────────────────────────────
frame_idx      = 0
preview_every  = max(1, total // 20)  # ~20장 미리보기

print('🚀 추론 시작...')
while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break
    if MAX_PREVIEW_FRAMES and frame_idx >= MAX_PREVIEW_FRAMES:
        break

    # 1. 얼굴 비식별화
    clean = anonymizer.blur(frame.copy())

    # 2. 사람 감지
    results    = person_model(clean, verbose=False)
    detections = sv.Detections.from_ultralytics(results[0])
    detections = detections[
        (detections.class_id == 0) &
        (detections.confidence >= CONFIDENCE_THRESHOLD)
    ]

    # 3. 추적
    detections = tracker.update(detections)

    # 4. 라인 크로싱 감지
    crossed_in, crossed_out = line_zone.trigger(detections)
    for i, (in_, out_) in enumerate(zip(crossed_in, crossed_out)):
        if in_ or out_:
            tid  = int(detections.tracker_id[i]) if detections.tracker_id is not None else -1
            conf = float(detections.confidence[i]) if detections.confidence is not None else 0.0
            if tid not in triggered_ids:
                triggered_ids.add(tid)
                event_log.append((frame_idx, tid, round(conf, 3)))
                print(f'  ⚠️  [프레임 {frame_idx:05d}] 라인 크로싱 감지 — Track #{tid}  conf={conf:.3f}')

    # 5. 어노테이션
    annotated = tracker.annotate(clean, detections)
    annotated = line_annotator.annotate(annotated, line_zone)

    # 6. 결과 저장
    writer.write(annotated)

    # 7. 미리보기
    if frame_idx % preview_every == 0:
        show_frame(annotated)
        print(f'  📸 프레임 {frame_idx}/{MAX_PREVIEW_FRAMES or total}')

    frame_idx += 1

cap.release()
writer.release()

print(f'\n✅ 추론 완료 — 총 {frame_idx}프레임 처리')
print(f'📁 결과 저장: {OUTPUT_PATH}')
print(f'🚨 감지된 이벤트: {len(event_log)}건')

## 6️⃣ 이벤트 요약

In [ ]:
if event_log:
    print(f'{'프레임':>8}  {'Track ID':>10}  {'신뢰도':>8}')
    print('-' * 34)
    for f_idx, tid, conf in event_log:
        print(f'{f_idx:>8}  {tid:>10}  {conf:>8.3f}')
else:
    print('감지된 라인 크로싱 이벤트 없음')

## 7️⃣ 결과 영상 미리보기 (선택)

In [ ]:
# 저장된 결과 영상의 첫 50프레임을 GIF로 미리보기
from IPython.display import HTML
import base64

cap2   = cv2.VideoCapture(OUTPUT_PATH)
frames = []
while len(frames) < 50:
    ret, f = cap2.read()
    if not ret: break
    frames.append(Image.fromarray(cv2.cvtColor(f, cv2.COLOR_BGR2RGB)))
cap2.release()

if frames:
    gif_buf = io.BytesIO()
    frames[0].save(
        gif_buf, format='GIF', save_all=True,
        append_images=frames[1:], loop=0, duration=60
    )
    b64 = base64.b64encode(gif_buf.getvalue()).decode()
    display(HTML(f'<img src="data:image/gif;base64,{b64}" style="max-width:100%"/>'))
    print(f'\n총 {len(frames)}프레임 미리보기')
else:
    print('미리볼 프레임이 없습니다.')